# **Relative occurances of 12mer & 15mer of absent peptides in CDS sequences**

In [ ]:
import shutil; shutil.rmtree("/content/positive_control_scatterplots", ignore_errors=True)

In [ ]:
# ============================================================================
# Missing Tetra/Pentapeptide Combinatorics vs. CDS Mapping — Scatterplots
# ============================================================================
import os
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats

# Install biopython if not already installed (uncomment if running fresh in Colab)
!pip install biopython -q
from Bio.Data import CodonTable

# --- 0. Mount Google Drive --------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# --- 1. Config: Paths, Styling, and FONT SIZES ------------------------------
CQWW_PATH  = "/content/CQWW_frames.csv"                                  # Tetrapeptides
PENTA_PATH = "/content/drive/MyDrive/final_pep_frames_tranls.csv"        # Pentapeptides
OUT_DIR    = "/content/peptide_combinatorics_scatterplots"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------
# FONT SIZE DICTIONARY: Change these values to adjust text!
# ---------------------------------------------------------
FONTS = {
    "suptitle": 18,       # Main figure heading at the very top
    "title": 15,          # Individual subplot titles
    "axis_label": 14,     # X and Y axis text
    "tick_label": 12,     # Numbers along the axes
    "stats_box": 14,      # The n, Spearman, and Pearson text inside the plot
    "bar_label": 18,      # The percentages floating above the bars
}

plt.rcParams.update({
    "figure.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False # Grid lines removed
})

COLORS = {"tetra": "#2E86AB", "penta": "#C1440E"}

# --- 2. Codon Degeneracy Engine -------------------------------------------
_degeneracy_cache = {}

def get_degeneracy_map(table_id):
    """Caches and returns the amino acid -> codons map for a specific table."""
    table_id = int(table_id)
    if table_id not in _degeneracy_cache:
        table = CodonTable.unambiguous_dna_by_id[table_id]
        aa_to_codons = defaultdict(list)
        for codon, aa in table.forward_table.items():
            aa_to_codons[aa].append(codon)
        _degeneracy_cache[table_id] = aa_to_codons
    return _degeneracy_cache[table_id]

def total_possible_nmers(peptide, table_id):
    """Calculates the theoretical max n-mers for a peptide under a given table."""
    deg_map = get_degeneracy_map(table_id)
    total = 1
    for aa in peptide:
        codons = deg_map.get(aa)
        if not codons:
            return np.nan
        total *= len(codons)
    return total

# --- 3. Data Loading & Standardization (With Semicolon Fix) ---------------
def clean_table(x):
    """Mimics pipeline logic: forces invalid or missing tables to 1."""
    if pd.isna(x):
        return 1
    x = str(x).strip()
    if x.endswith(".0"):
        x = x.replace(".0", "")
    try:
        return int(x)
    except Exception:
        return 1

def load_and_summarize(path, expected_len, ptype, match_col):
    print(f"Loading {ptype} data from {path}...")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    df["peptide"] = df["peptide"].astype(str).str.strip().str.upper()
    df["transl_table"] = df["transl_table"].apply(clean_table)

    df = df[df["peptide"].str.len() == expected_len].copy()

    if match_col not in df.columns:
        raise KeyError(f"Column '{match_col}' not found. Available columns: {df.columns.tolist()}")

    df[match_col] = df[match_col].astype(str).str.split(";")
    df = df.explode(match_col)
    df[match_col] = df[match_col].str.strip().str.upper()

    rows = []
    for (peptide, table_id), g in df.groupby(["peptide", "transl_table"]):
        total_possible = total_possible_nmers(peptide, table_id)
        distinct_mapped = g[match_col].nunique()

        if pd.notna(total_possible) and total_possible > 0:
            fraction = distinct_mapped / total_possible

            rows.append({
                "peptide": peptide,
                "transl_table": table_id,
                "total_possible": total_possible,
                "distinct_mapped": distinct_mapped,
                "fraction_mapped": fraction
            })

    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(os.path.join(OUT_DIR, f"{ptype}_summary_data.csv"), index=False)
    return summary_df

tetra_df = load_and_summarize(CQWW_PATH, 4, "tetra", "matched_15mer")
penta_df = load_and_summarize(PENTA_PATH, 5, "penta", "matched_15mer")

# --- 4. Plotting Engine ---------------------------------------------------
def plot_scatter_with_stats(df, x_col, y_col, ptype, ax, title, ylabel, is_log_x=False, is_log_y=False):
    """Plots a scatterplot and embeds correlation stats inside the plot area."""
    if df.empty or len(df) < 3:
        ax.text(0.5, 0.5, "Insufficient Data", ha='center', va='center', transform=ax.transAxes, fontsize=FONTS["stats_box"])
        return

    x = df[x_col].values
    y = df[y_col].values
    color = COLORS[ptype]

    # Scatter plot
    ax.scatter(x, y, color=color, alpha=0.6, edgecolor="white", s=80)

    # Calculate Correlations
    spearman_r, _ = stats.spearmanr(x, y)

    valid = (x > 0)
    if is_log_y:
        valid = valid & (y > 0)

    x_val = np.log10(x[valid]) if is_log_x else x[valid]
    y_val = np.log10(y[valid]) if is_log_y else y[valid]

    if len(x_val) > 1:
        pearson_r, _ = stats.pearsonr(x_val, y_val)
    else:
        pearson_r = np.nan

    # Apply X-Axis Scaling
    if is_log_x:
        ax.set_xscale('log')
        # Mathtext formatter ensures 10^0, 10^1 format is displayed
        ax.xaxis.set_major_formatter(ticker.LogFormatterMathtext())
    else:
        # Explicit Linear Boundaries for Tetrapeptides
        ax.set_xlim(2, 20)
        ax.set_xticks(np.arange(2, 22, 2))

    # Apply Y-Axis Scaling
    if is_log_y:
        ax.set_yscale('log')
        ax.yaxis.set_major_formatter(ticker.LogFormatterMathtext())
    elif "Fraction" in ylabel:
        # Pinned exactly to 0 base per request
        ax.set_ylim(0, 1.05)

    # Add breathing room around points so they don't rest exactly on spines if not pinned
    if "Fraction" not in ylabel:
        ax.margins(0.15)

    # Apply Font Sizes from the Dictionary
    ax.set_title(title, pad=15, fontweight='bold', fontsize=FONTS["title"])
    ax.set_xlabel("Theoretically Possible n-mers", fontweight='bold', fontsize=FONTS["axis_label"])
    ax.set_ylabel(ylabel, fontweight='bold', fontsize=FONTS["axis_label"])
    ax.tick_params(axis='both', which='major', labelsize=FONTS["tick_label"])

    stats_text = (
        f"n = {len(df):,}\n"
        f"Spearman r = {spearman_r:.2f}\n"
        f"Pearson r = {pearson_r:.2f}"
    )

    # Guaranteed empty-space placement
    if "Distinct" in title:
        # Left plot: Distinct Mapped vs Possible. Top-left is always biologically empty.
        box_x, box_y = 0.05, 0.95
        va, ha = 'top', 'left'
    else:
        # Right plot: Fraction Mapped vs Possible. Top-right is always biologically empty.
        box_x, box_y = 0.95, 0.95
        va, ha = 'top', 'right'

    ax.text(box_x, box_y, stats_text, transform=ax.transAxes, fontsize=FONTS["stats_box"],
            verticalalignment=va, horizontalalignment=ha,
            bbox=dict(boxstyle="round,pad=0.5", facecolor='white', alpha=0.8, edgecolor='#CCCCCC'))

def create_figure_for_group(df, ptype, name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    # Conditional logic: only pentapeptides get logarithmic scales
    use_log = True if ptype == "penta" else False

    plot_scatter_with_stats(
        df=df, x_col="total_possible", y_col="distinct_mapped", ptype=ptype, ax=axes[0],
        title=f"{name}: Possible vs. Distinct Mapped",
        ylabel="Distinct n-mer Variants Mapped",
        is_log_x=use_log,
        is_log_y=use_log
    )

    plot_scatter_with_stats(
        df=df, x_col="total_possible", y_col="fraction_mapped", ptype=ptype, ax=axes[1],
        title=f"{name}: Possible vs. Fraction Mapped",
        ylabel="Fraction of Possible n-mers Mapped",
        is_log_x=use_log,
        is_log_y=False
    )

    fig.suptitle(f"Combinatorial Space Analysis: Missing {name}", fontsize=FONTS["suptitle"], fontweight='bold', y=1.05)
    fig.tight_layout()
    output_path = os.path.join(OUT_DIR, f"{ptype}_scatterplots.png")
    fig.savefig(output_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {output_path}")

def plot_total_representation(tetra_df, penta_df):
    """Calculates and plots the aggregate percentage of combinatorial space used, including raw counts."""
    fig, ax = plt.subplots(figsize=(10, 8))

    # Calculate aggregates
    tetra_possible = int(tetra_df["total_possible"].sum())
    tetra_mapped = int(tetra_df["distinct_mapped"].sum())
    tetra_pct = (tetra_mapped / tetra_possible * 100) if tetra_possible > 0 else 0

    penta_possible = int(penta_df["total_possible"].sum())
    penta_mapped = int(penta_df["distinct_mapped"].sum())
    penta_pct = (penta_mapped / penta_possible * 100) if penta_possible > 0 else 0

    labels = ['12mers', '15mers']
    percentages = [tetra_pct, penta_pct]
    mapped_counts = [tetra_mapped, penta_mapped]
    possible_counts = [tetra_possible, penta_possible]
    colors = [COLORS['tetra'], COLORS['penta']]

    bars = ax.bar(labels, percentages, color=colors, alpha=0.85, width=0.5, edgecolor="white")

    for bar, pct, mapped, possible in zip(bars, percentages, mapped_counts, possible_counts):
        label_text = f"{mapped:,} / {possible:,}\n({pct:.2f}%)"

        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + (max(percentages) * 0.02),
                label_text, ha='center', va='bottom', fontweight='bold', fontsize=FONTS["bar_label"])

    ax.set_ylabel("Aggregate Representation (%)", fontweight='bold', fontsize=FONTS["axis_label"])
    ax.set_title("Total Combinatorial Space Utilized by missing peptides", pad=15, fontweight='bold', fontsize=FONTS["suptitle"])
    ax.tick_params(axis='both', which='major', labelsize=FONTS["tick_label"])
    ax.set_ylim(0, max(percentages) * 1.35 if max(percentages) > 0 else 100)

    fig.tight_layout()
    output_path = os.path.join(OUT_DIR, "total_representation_barplot.png")
    fig.savefig(output_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {output_path}")

# --- 5. Generate Independent Figures ----------------------------------------
create_figure_for_group(tetra_df, "tetra", "Tetrapeptides")
create_figure_for_group(penta_df, "penta", "Pentapeptides")
plot_total_representation(tetra_df, penta_df)

print("\nAnalysis complete. High-resolution plots and data tables are in the output directory.")

### **Relative occurances of 12mer & 15mer of present peptides in CDS sequences**

In [ ]:
# ============================================================================
# POSITIVE CONTROL: Present Tetra/Pentapeptide Combinatorics vs. CDS Mapping
# ============================================================================
import os
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats

# Install biopython if not already installed
# !pip install biopython -q
from Bio.Data import CodonTable

# --- 0. Mount Google Drive --------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# --- 1. Config: Paths, Styling, and FONT SIZES ------------------------------
# UPDATE THESE PATHS TO YOUR POSITIVE CONTROL FILES
TETRA_PATH = "/content/WWQC_CDS_specific_hits_all.csv"
PENTA_PATH = "/content/drive/MyDrive/MASTER_reverse_pentapeptide_CDS_hits_full_details.csv"
OUT_DIR    = "/content/positive_control_scatterplots"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------
# FONT SIZE DICTIONARY: Change these values to adjust text!
# ---------------------------------------------------------
FONTS = {
    "suptitle": 18,       # Main figure heading at the very top
    "title": 15,          # Individual subplot titles
    "axis_label": 14,     # X and Y axis text
    "tick_label": 12,     # Numbers along the axes
    "stats_box": 14,      # The n, Spearman, and Pearson text inside the plot
    "bar_label": 18,      # The percentages floating above the bars
}

plt.rcParams.update({
    "figure.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False # Grid lines removed
})

# distinct colors for the positive control group to avoid confusion with missing peptides
COLORS = {"tetra": "#2ca02c", "penta": "#9467bd"} # Green and Purple

# --- 2. Codon Degeneracy Engine -------------------------------------------
_degeneracy_cache = {}

def get_degeneracy_map(table_id):
    table_id = int(table_id)
    if table_id not in _degeneracy_cache:
        table = CodonTable.unambiguous_dna_by_id[table_id]
        aa_to_codons = defaultdict(list)
        for codon, aa in table.forward_table.items():
            aa_to_codons[aa].append(codon)
        _degeneracy_cache[table_id] = aa_to_codons
    return _degeneracy_cache[table_id]

def total_possible_nmers(peptide, table_id):
    deg_map = get_degeneracy_map(table_id)
    total = 1
    for aa in peptide:
        codons = deg_map.get(aa)
        if not codons:
            return np.nan
        total *= len(codons)
    return total

# --- 3. Dynamic Data Loading (Handles varying column names) ---------------
def clean_table(x):
    """Mimics pipeline logic: forces invalid or missing tables to 1."""
    if pd.isna(x):
        return 1
    x = str(x).strip()
    if x.endswith(".0"):
        x = x.replace(".0", "")
    try:
        return int(x)
    except Exception:
        return 1

def load_and_summarize(path, expected_len, ptype, col_pep, col_match, col_table):
    print(f"Loading {ptype} data from {path}...")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    df[col_pep] = df[col_pep].astype(str).str.strip().str.upper()
    df[col_table] = df[col_table].apply(clean_table)

    df = df[df[col_pep].str.len() == expected_len].copy()

    if col_match not in df.columns:
        raise KeyError(f"Column '{col_match}' not found. Available columns: {df.columns.tolist()}")

    df[col_match] = df[col_match].astype(str).str.split(";")
    df = df.explode(col_match)
    df[col_match] = df[col_match].str.strip().str.upper()

    rows = []
    for (peptide, table_id), g in df.groupby([col_pep, col_table]):
        total_possible = total_possible_nmers(peptide, table_id)
        distinct_mapped = g[col_match].nunique()

        if pd.notna(total_possible) and total_possible > 0:
            fraction = distinct_mapped / total_possible

            rows.append({
                "peptide": peptide,
                "transl_table": table_id,
                "total_possible": total_possible,
                "distinct_mapped": distinct_mapped,
                "fraction_mapped": fraction
            })

    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(os.path.join(OUT_DIR, f"{ptype}_present_summary.csv"), index=False)
    return summary_df

tetra_df = load_and_summarize(TETRA_PATH, 4, "tetra", "peptide", "matched_12mer", "cds_transl_table")
penta_df = load_and_summarize(PENTA_PATH, 5, "penta", "reverse_peptide", "matched_reverse_15mer", "reverse_cds_transl_table")

# --- 4. Plotting Engine ---------------------------------------------------
def plot_scatter_with_stats(df, x_col, y_col, ptype, ax, title, ylabel, is_log_x=False, is_log_y=False):
    if df.empty or len(df) < 3:
        ax.text(0.5, 0.5, "Insufficient Data", ha='center', va='center', transform=ax.transAxes, fontsize=FONTS["stats_box"])
        return

    x = df[x_col].values
    y = df[y_col].values
    color = COLORS[ptype]

    ax.scatter(x, y, color=color, alpha=0.6, edgecolor="white", s=80)

    spearman_r, _ = stats.spearmanr(x, y)

    valid = (x > 0)
    if is_log_y:
        valid = valid & (y > 0)

    x_val = np.log10(x[valid]) if is_log_x else x[valid]
    y_val = np.log10(y[valid]) if is_log_y else y[valid]

    if len(x_val) > 1:
        pearson_r, _ = stats.pearsonr(x_val, y_val)
    else:
        pearson_r = np.nan

    # Apply X-Axis Scaling
    if is_log_x:
        ax.set_xscale('log')
        ax.xaxis.set_major_formatter(ticker.LogFormatterMathtext())
    else:
        # Explicit Linear Boundaries for Tetrapeptides
        ax.set_xlim(2, 20)
        ax.set_xticks(np.arange(2, 22, 2))

    # Apply Y-Axis Scaling
    if is_log_y:
        ax.set_yscale('log')
        ax.yaxis.set_major_formatter(ticker.LogFormatterMathtext())
    elif "Fraction" in ylabel:
        # Pinned exactly to 0 base
        ax.set_ylim(0, 1.05)

    # Add breathing room around points
    if "Fraction" not in ylabel:
        ax.margins(0.15)

    # Apply Font Sizes from the Dictionary
    ax.set_title(title, pad=15, fontweight='bold', fontsize=FONTS["title"])
    ax.set_xlabel("Theoretically Possible n-mers", fontweight='bold', fontsize=FONTS["axis_label"])
    ax.set_ylabel(ylabel, fontweight='bold', fontsize=FONTS["axis_label"])
    ax.tick_params(axis='both', which='major', labelsize=FONTS["tick_label"])

    stats_text = (
        f"n = {len(df):,}\n"
        f"Spearman r = {spearman_r:.2f}\n"
        f"Pearson r = {pearson_r:.2f}"
    )

    # Guaranteed empty-space placement
    if "Distinct" in title:
        box_x, box_y = 0.05, 0.95
        va, ha = 'top', 'left'
    else:
        box_x, box_y = 0.95, 0.95
        va, ha = 'top', 'right'

    ax.text(box_x, box_y, stats_text, transform=ax.transAxes, fontsize=FONTS["stats_box"],
            verticalalignment=va, horizontalalignment=ha,
            bbox=dict(boxstyle="round,pad=0.5", facecolor='white', alpha=0.8, edgecolor='#CCCCCC'))

def create_figure_for_group(df, ptype, name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    # Conditional logic: only pentapeptides get logarithmic scales
    use_log = True if ptype == "penta" else False

    plot_scatter_with_stats(
        df=df, x_col="total_possible", y_col="distinct_mapped", ptype=ptype, ax=axes[0],
        title=f"{name}: Possible vs. Distinct Mapped",
        ylabel="Distinct n-mer Variants Mapped",
        is_log_x=use_log,
        is_log_y=use_log
    )

    plot_scatter_with_stats(
        df=df, x_col="total_possible", y_col="fraction_mapped", ptype=ptype, ax=axes[1],
        title=f"{name}: Possible vs. Fraction Mapped",
        ylabel="Fraction of Possible n-mers Mapped",
        is_log_x=use_log,
        is_log_y=False
    )

    fig.suptitle(f"Positive Control Analysis: Present {name}", fontsize=FONTS["suptitle"], fontweight='bold', y=1.05)
    fig.tight_layout()
    output_path = os.path.join(OUT_DIR, f"{ptype}_present_scatterplots.png")
    fig.savefig(output_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {output_path}")

def plot_total_representation(tetra_df, penta_df):
    fig, ax = plt.subplots(figsize=(10, 8))

    tetra_possible = int(tetra_df["total_possible"].sum())
    tetra_mapped = int(tetra_df["distinct_mapped"].sum())
    tetra_pct = (tetra_mapped / tetra_possible * 100) if tetra_possible > 0 else 0

    penta_possible = int(penta_df["total_possible"].sum())
    penta_mapped = int(penta_df["distinct_mapped"].sum())
    penta_pct = (penta_mapped / penta_possible * 100) if penta_possible > 0 else 0

    labels = ['12mers', '15mers']
    percentages = [tetra_pct, penta_pct]
    mapped_counts = [tetra_mapped, penta_mapped]
    possible_counts = [tetra_possible, penta_possible]
    colors = [COLORS['tetra'], COLORS['penta']]


    bars = ax.bar(labels, percentages, color=colors, alpha=0.85, width=0.5, edgecolor="white")

    for bar, pct, mapped, possible in zip(bars, percentages, mapped_counts, possible_counts):
        label_text = f"{mapped:,} / {possible:,}\n({pct:.2f}%)"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + (max(percentages) * 0.02),
                label_text, ha='center', va='bottom', fontweight='bold', fontsize=FONTS["bar_label"])

    ax.set_ylabel("Aggregate Representation (%)", fontweight='bold', fontsize=FONTS["axis_label"])
    ax.set_title("Total Combinatorial Space Utilized by present peptides", pad=15, fontweight='bold', fontsize=FONTS["suptitle"])
    ax.tick_params(axis='both', which='major', labelsize=FONTS["tick_label"])
    ax.set_ylim(0, max(percentages) * 1.35 if max(percentages) > 0 else 100)

    fig.tight_layout()
    output_path = os.path.join(OUT_DIR, "present_total_representation_barplot.png")
    fig.savefig(output_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {output_path}")

# --- 5. Generate Independent Figures ----------------------------------------
create_figure_for_group(tetra_df, "tetra", "Tetrapeptides")
create_figure_for_group(penta_df, "penta", "Pentapeptides")
plot_total_representation(tetra_df, penta_df)

print("\nPositive Control Analysis complete. High-resolution plots are in the output directory.")

# **Phylogenetic Constraint: Translation Table Heatmap (Missing vs. Present)**

In [ ]:
# ============================================================================
# Phylogenetic Constraint: Independent Heatmaps (Tetra vs Penta)
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Config: Paths & Styling ---------------------------------------------
MISSING_TETRA = "/content/peptide_combinatorics_scatterplots/tetra_summary_data.csv"
MISSING_PENTA = "/content/peptide_combinatorics_scatterplots/penta_summary_data.csv"
PRESENT_TETRA = "/content/positive_control_scatterplots/tetra_present_summary.csv"
PRESENT_PENTA = "/content/positive_control_scatterplots/penta_present_summary.csv"

OUT_DIR = "/content/phylogenetic_heatmaps"
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 300,
    "axes.titlesize": 20,    # Increased for hierarchy
    "axes.labelsize": 18,    # Increased X/Y axis labels
    "xtick.labelsize": 14,   # Slightly increased for readability
    "ytick.labelsize": 14,   # Slightly increased for readability
})

# NCBI Translation Table Taxonomy Mapping
TABLE_NAMES = {
    1: "1: Standard (Eukaryotes)",
    2: "2: Vertebrate Mitochondrial",
    3: "3: Yeast Mitochondrial",
    4: "4: Mold, Protozoan, Coelenterate Mito",
    5: "5: Invertebrate Mitochondrial",
    6: "6: Ciliate, Dasycladacean, Hexamita",
    9: "9: Echinoderm & Flatworm Mito",
    10: "10: Euplotid Nuclear",
    11: "11: Bacterial, Archaeal & Plant Plastid",
    12: "12: Alternative Yeast Nuclear",
    13: "13: Ascidian Mitochondrial",
    14: "14: Alternative Flatworm Mitochondrial",
    15: "15: Blepharisma Macronuclear",
    16: "16: Chlorophycean Mitochondrial",
    21: "21: Trematode Mitochondrial",
    22: "22: Scenedesmus obliquus Mitochondrial",
    23: "23: Thraustochytrium Mitochondrial",
    24: "24: Pterobranchia Mitochondrial",
    25: "25: Candidate Division SR1 & Gracilibacteria",
    28: "28: Condylostoma Nuclear",
    33: "33: Cephalodiscidae Mitochondrial"
}

# --- 2. Data Processing Engine ----------------------------------------------
def get_table_averages(filepath, column_name):
    """Loads summary data and calculates the mean fraction mapped per translation table."""
    if not os.path.exists(filepath):
        print(f"[Warning] File not found: {filepath}")
        return pd.Series(dtype=float, name=column_name)

    df = pd.read_csv(filepath)
    if "transl_table" not in df.columns or "fraction_mapped" not in df.columns:
        print(f"[Warning] Required columns missing in {filepath}")
        return pd.Series(dtype=float, name=column_name)

    return df.groupby("transl_table")["fraction_mapped"].mean().rename(column_name)

# Extract averages for all four groups
m_tetra_avg = get_table_averages(MISSING_TETRA, "Missing 12mers")
p_tetra_avg = get_table_averages(PRESENT_TETRA, "Present 12mers")
m_penta_avg = get_table_averages(MISSING_PENTA, "Missing 15mers")
p_penta_avg = get_table_averages(PRESENT_PENTA, "Present 15mers")

# --- 3. Heatmap Generation Function -----------------------------------------
def generate_independent_heatmap(group_df, title, output_filename):
    """Generates a dynamically sized heatmap for a specific peptide group."""

    # Drop rows where ALL columns are NaN for this specific dataframe
    clean_df = group_df.dropna(how='all').sort_index()

    if clean_df.empty:
        print(f"[Skipping] No data available for {title}")
        return

    # Map the index (integer IDs) to descriptive phylogenetic names
    clean_df.index = [TABLE_NAMES.get(int(idx), f"{int(idx)}: Unknown/Unassigned") for idx in clean_df.index]

    # Dynamically scale figure height based on the number of surviving rows
    fig_height = max(4.0, len(clean_df) * 0.8)
    fig, ax = plt.subplots(figsize=(8, fig_height))

    sns.heatmap(
        clean_df,
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",
        linewidths=1.5,
        linecolor='white',
        cbar_kws={'label': 'Mean Fraction Mapped'},
        ax=ax,
        vmin=0.0,
        vmax=1.0,
        annot_kws={"size": 16, "weight": "bold"}
    )

    # Styling and Labels
    ax.set_title(title, pad=20, fontweight='bold', fontsize=20)
    ax.set_xlabel("Peptide Groups", fontweight='bold', labelpad=15, fontsize=18)
    ax.set_ylabel("Translation Tables (Taxonomic Domains)", fontweight='bold', labelpad=15, fontsize=18)

    plt.xticks(rotation=15, ha='right')
    plt.yticks(rotation=0)

    fig.tight_layout()

    # Save output
    output_path = os.path.join(OUT_DIR, output_filename)
    fig.savefig(output_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {output_path} (Rendered {clean_df.shape[0]} tables)")

# --- 4. Matrix Assembly & Execution -----------------------------------------
# Group the tetrapeptides and pentapeptides independently
tetra_df = pd.concat([m_tetra_avg, p_tetra_avg], axis=1)
penta_df = pd.concat([m_penta_avg, p_penta_avg], axis=1)

# Generate Plot 1
generate_independent_heatmap(
    group_df=tetra_df,
    title="Translation table wise\nTetrapeptide mapped fraction",
    output_filename="heatmap_tetrapeptides.png"
)

# Generate Plot 2
generate_independent_heatmap(
    group_df=penta_df,
    title="Translation table wise\nPentapeptide mapped fraction",
    output_filename="heatmap_pentapeptides.png"
)

print("\nIndependent heatmap generation complete!")

# **Pentapeptide Frame 0 Probability: Theoretical vs. Actual Abundance**

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Install & Import Biopython (Required for Theoretical Space) ---------
try:
    from Bio.Data import CodonTable
except ImportError:
    import subprocess
    print("Installing Biopython...")
    subprocess.check_call(["pip", "install", "biopython"])
    from Bio.Data import CodonTable

# --- 2. Mount Google Drive & Environment Setup ------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!\n")
except ImportError:
    print("[Note] Not running in Google Colab. Skipping Drive mount.\n")

# File Paths (Pentapeptides only)
MISSING_PENTA_RAW = "/content/drive/MyDrive/Pentapeptide_frames_file/final_pep_frames_tranls_repaired.csv"
PRESENT_PENTA_RAW = "/content/drive/MyDrive/MASTER_reverse_pentapeptide_CDS_hits_full_details.csv"

OUT_DIR = "/content/frame_analysis_pentapeptide"
os.makedirs(OUT_DIR, exist_ok=True)

# Styling Config for Manuscript-Quality Figures
plt.rcParams.update({
    "figure.dpi": 300,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False
})

COLORS = {"Missing": "#C1440E", "Present": "#9467bd"}

# --- 3. Unified Data Processing Engine --------------------------------------
def get_combinatorial_space(peptide, table_id):
    """Calculates the theoretical number of synonymous n-mers for a peptide."""
    try:
        table = CodonTable.unambiguous_dna_by_id[int(table_id)]
    except:
        table = CodonTable.unambiguous_dna_by_id[1]

    degeneracy_map = {}
    for codon, aa in table.forward_table.items():
        degeneracy_map[aa] = degeneracy_map.get(aa, 0) + 1

    space = 1
    for aa in str(peptide).upper():
        space *= degeneracy_map.get(aa, 1)
    return space

def process_unified_dataset(filepath, group_name, peptide_col, table_col, frame_calc_type, frame_source_col):
    """Extracts both total mapping frequency and Frame 0 presence in one pass."""
    if not os.path.exists(filepath):
        print(f"[Warning] File not found: {filepath}")
        return pd.DataFrame()

    # low_memory=False prevents DtypeWarnings on large files
    df = pd.read_csv(filepath, low_memory=False)

    # Calculate Frame 0 mapping
    if frame_calc_type == 'shift':
        clean_shift = df[frame_source_col].astype(str).str.strip()
        frame_0_conditions = ['0', '0.0', 'transl_table_issue', 'transl_table_error']
        df['is_frame_0'] = clean_shift.isin(frame_0_conditions)

    elif frame_calc_type == 'start_0based':
        numeric_start = pd.to_numeric(df[frame_source_col], errors='coerce')
        df['is_frame_0'] = (numeric_start % 3) == 0

    df = df.rename(columns={peptide_col: 'peptide', table_col: 'transl_table'})
    df = df.dropna(subset=['peptide', 'transl_table'])

    # Group by peptide to count total occurrences (mapped_count) and if Frame 0 exists
    grouped = df.groupby(['peptide', 'transl_table']).agg(
        mapped_count=('is_frame_0', 'count'),
        has_frame_0=('is_frame_0', 'any')
    ).reset_index()

    # Finalize columns for plotting
    grouped['frame_0_binary'] = grouped['has_frame_0'].astype(int)
    grouped['total_possible'] = grouped.apply(lambda row: get_combinatorial_space(row['peptide'], row['transl_table']), axis=1)
    grouped['Group'] = group_name

    return grouped

# --- 4. Matrix Assembly -----------------------------------------------------
print("Processing Pentapeptide Datasets...")
m_penta = process_unified_dataset(
    MISSING_PENTA_RAW, "Missing", "peptide", "transl_table",
    frame_calc_type="shift", frame_source_col="shift"
)
p_penta = process_unified_dataset(
    PRESENT_PENTA_RAW, "Present", "reverse_peptide", "reverse_cds_transl_table",
    frame_calc_type="start_0based", frame_source_col="reverse_start_0based"
)
df_penta = pd.concat([m_penta, p_penta], ignore_index=True)

# --- 5. Universal Plotting Function -----------------------------------------
def plot_regression(ax, df, x_column, title, x_label):
    if df.empty:
        return

    # Plot missing trend
    sns.regplot(
        data=df[df['Group'] == 'Missing'],
        x=x_column,
        y='frame_0_binary',
        logistic=True,
        ci=95,
        scatter=False,
        color=COLORS["Missing"],
        label='Missing Trend',
        ax=ax,
        line_kws={'linewidth': 3}
    )

    # Plot present trend
    sns.regplot(
        data=df[df['Group'] == 'Present'],
        x=x_column,
        y='frame_0_binary',
        logistic=True,
        ci=95,
        scatter=False,
        color=COLORS["Present"],
        label='Present Trend',
        ax=ax,
        line_kws={'linewidth': 3, 'linestyle': '--'}
    )

    ax.set_title(title, pad=15, fontweight='bold')
    ax.set_xlabel(x_label, fontweight='bold')
    ax.set_ylabel("Probability of Frame 0 Mapping", fontweight='bold')

    # Apply log scaling and explicit left boundary to prevent log(0) calculation errors
    ax.set_xscale('log')
    ax.set_xlim(left=0.5)
    ax.set_ylim(-0.05, 1.05)

    # Adjust legend position based on which plot it is to avoid covering the lines
    legend_loc = "upper right" if x_column == "total_possible" else "lower right"
    ax.legend(loc=legend_loc, framealpha=0.9)

# --- 6. Execution & Saving --------------------------------------------------
print("\nGenerating Unified Figure...")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Theoretical Possibilities (The Spike)
plot_regression(
    axes[0], df_penta,
    x_column="total_possible",
    title="Pentapeptides: Theoretical Flexibility vs Frame 0",
    x_label="Theoretically Possible n-mers"
)

# Panel 2: Mapped Frequency (The Flatline)
plot_regression(
    axes[1], df_penta,
    x_column="mapped_count",
    title="Pentapeptides: Genomic Abundance vs Frame 0",
    x_label="Total Mapped Occurrences in CDS"
)

fig.suptitle("Evolutionary Censorship of Missing Pentapeptides", fontsize=18, fontweight='bold', y=1.05)
fig.tight_layout()

output_path = os.path.join(OUT_DIR, "unified_pentapeptide_analysis.png")
fig.savefig(output_path, bbox_inches="tight")
plt.close(fig)
print(f"Analysis complete! Unified plot saved to: {output_path}")

#